# Tatar (tat) — Full NLP Pipeline with Custom Stanza Models

Tatar is written in Cyrillic (primary). The Zamanälif Latin alphabet provides an alternative script. TurkicNLP offers Production-quality Apertium FST morphology, custom-trained Stanza neural models for POS tagging, lemmatisation, and dependency parsing, and bidirectional Cyrillic↔Latin (Zamanälif) transliteration.

In [ ]:
# Install TurkicNLP
# pip install turkicnlp          # core (tokenization, transliteration)
# pip install "turkicnlp[stanza]"  # adds POS, lemma, depparse, NER
# pip install "turkicnlp[nllb]"    # adds cross-lingual embeddings + translation
# pip install "turkicnlp[all]"     # all optional dependencies

In [ ]:
import turkicnlp
from turkicnlp import Pipeline

## 1. Download Models

In [ ]:
turkicnlp.download('tat')

## 2. Cyrillic ↔ Latin Transliteration (Zamanälif)

In [ ]:
from turkicnlp.scripts import Script
from turkicnlp.scripts.detector import detect_script
from turkicnlp.scripts.transliterator import Transliterator

print("=" * 70)
print("TATAR COMPREHENSIVE TRANSLITERATION")
print("=" * 70)
print()

cyrl = "Мин мәктәпкә барам."
print(f"Original (Cyrillic, primary): {cyrl}")
print()

# Direction 1: Cyrillic → Turkic Common Alphabet (Latin, Zamanälif)
print("1. Cyrillic → Turkic Common Alphabet (Latin, Zamanälif):")
try:
    t1 = Transliterator("tat", source=Script.CYRILLIC, target=Script.COMMON_TURKIC)
    common = t1.transliterate(cyrl)
    print(f"   {common}")
except Exception as e:
    print(f"   ⚠ Not supported: {e}")
print()

# Direction 2: Turkic Common → Cyrillic (reverse)
print("2. Turkic Common (Latin) → Cyrillic:")
try:
    t2 = Transliterator("tat", source=Script.COMMON_TURKIC, target=Script.CYRILLIC)
    back_to_cyrl = t2.transliterate(common if 'common' in locals() else "Min məktəpkə baram.")
    print(f"   {back_to_cyrl}")
    print(f"   ✓ Round-trip match: {cyrl == back_to_cyrl}")
except Exception as e:
    print(f"   ⚠ Not supported: {e}")
print()

# Direction 3: Cyrillic → Latin
print("3. Cyrillic → Latin (Zamanälif, explicit):")
try:
    t3 = Transliterator("tat", source=Script.CYRILLIC, target=Script.LATIN)
    latin = t3.transliterate(cyrl)
    print(f"   {latin}")
except Exception as e:
    print(f"   ⚠ Not supported: {e}")
print()

# Direction 4: Latin → Cyrillic
print("4. Latin (Zamanälif) → Cyrillic:")
try:
    t4 = Transliterator("tat", source=Script.LATIN, target=Script.CYRILLIC)
    back_to_cyrl_explicit = t4.transliterate(latin if 'latin' in locals() else "Min məktəpkə baram.")
    print(f"   {back_to_cyrl_explicit}")
except Exception as e:
    print(f"   ⚠ Not supported: {e}")
print()

print("=" * 70)
print("Tatar Scripts:")
print("  • Cyrillic (primary, Soviet legacy)")
print("  • Latin - Zamanälif (new Tatar Latin, COMMON_TURKIC standard)")
print("=" * 70)

## 3. Morphological Analysis (Apertium FST — Production quality)

In [ ]:
nlp = Pipeline(
    "tat",
    processors=["tokenize", "morph"],
    morph_backend="apertium",
    script="Cyrl",
)
doc = nlp("Мин мәктәпкә барам.")
for w in doc.words:
    print(f"{w.text:<18} lemma={w.lemma:<12} feats={w.feats}")

## 4. POS Tagging, Lemmatisation, and Dependency Parsing (Custom Stanza)

TurkicNLP includes custom-trained Stanza models for Tatar, providing POS tagging, lemmatisation, and dependency parsing.

In [ ]:
nlp_parse = Pipeline(
    "tat",
    processors=["tokenize", "pos", "lemma", "depparse"],
    script="Cyrl",
)

doc = nlp_parse("Бер журнал бу ай санында аның тормышын микроскоп астына ала.")
print(f"{'Word':<20} {'UPOS':<8} {'Lemma':<20} {'Head':<5} {'Deprel'}")
print("-" * 60)
for w in doc.words:
    print(f"{w.text:<20} {w.upos:<8} {w.lemma:<20} {w.head!s:<5} {w.deprel}")

## 5. Full Pipeline with CoNLL-U Export

In [ ]:
nlp_full = Pipeline(
    "tat",
    processors=["tokenize", "morph", "pos", "lemma", "depparse"],
    morph_backend="apertium",
    script="Cyrl",
)
doc = nlp_full("Татарстан Россия Федерациясе составындагы республика.")
print(doc.to_conllu())

## 6. Translation

In [ ]:
turkicnlp.download("tat", processors=["translate"])
trans = Pipeline("tat", processors=["translate"], translate_tgt_lang="rus_Cyrl")
doc = trans("Татарстан Россия Федерациясе составындагы республика.")
print("RU:", doc.translation)